# Phase 4: Feature Engineering
Predictive Maintenance of Industrial Equipment using IoT Sensor Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/cleaned_sensor_data.csv', parse_dates=['Timestamp'])
df = df.sort_values(['Machine_ID','Timestamp']).reset_index(drop=True)
df.shape

## 4.1 Rolling average of Temperature and Vibration (per machine, 6-hour window)
A single hourly reading is noisy. A rolling average smooths short-term noise and reveals the real trend -- the same idea used in the earlier RUL project, applied here to a classification task.

In [ ]:
window = 6
for col in ['Temperature', 'Vibration', 'Pressure', 'RPM']:
    df[f'{col}_roll_mean'] = df.groupby('Machine_ID')[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
df[['Machine_ID','Timestamp','Temperature','Temperature_roll_mean']].head()

## 4.2 Maximum/minimum sensor readings over the rolling window

In [ ]:
for col in ['Temperature', 'Vibration']:
    df[f'{col}_roll_max'] = df.groupby('Machine_ID')[col].transform(lambda x: x.rolling(window, min_periods=1).max())
    df[f'{col}_roll_min'] = df.groupby('Machine_ID')[col].transform(lambda x: x.rolling(window, min_periods=1).min())
df[['Temperature_roll_max','Temperature_roll_min','Vibration_roll_max','Vibration_roll_min']].describe()

## 4.3 Sensor deviation from historical operating baseline
The baseline uses only readings available before the current timestamp. The first reading for each machine has deviation 0 because no earlier history exists. This avoids using future readings to define an earlier baseline.


In [ ]:
# Past-only expanding median baseline, shifted by one reading (no future values).
def add_deviation(df, col):
    past_baseline = df.groupby('Machine_ID')[col].transform(
        lambda x: x.expanding(min_periods=1).median().shift(1)
    )
    past_baseline = past_baseline.fillna(df[col])  # first reading: deviation 0
    df[f'{col}_deviation'] = df[col] - past_baseline
    return df

for col in ['Temperature','Vibration','Pressure','RPM']:
    df = add_deviation(df, col)
df[['Temperature_deviation','Vibration_deviation']].describe()


## 4.4 Operating-hours bands

In [ ]:
df['OpHours_Band'] = pd.cut(df['Operating_Hours'], bins=[0,1000,2000,3000,4000,6000],
                            labels=['0-1000','1000-2000','2000-3000','3000-4000','4000+'])
df['OpHours_Band'].value_counts()

## 4.5 Machine age groups

In [ ]:
df['Age_Group'] = pd.cut(df['Machine_Age'], bins=[0,3,6,9,15],
                         labels=['New(0-3y)','Mid(3-6y)','Old(6-9y)','VeryOld(9y+)'])
df['Age_Group'].value_counts()

## 4.6 Maintenance frequency (relative to machine age)

In [ ]:
df['Maintenance_Frequency'] = df['Maintenance_Count'] / df['Machine_Age'].replace(0,1)
df[['Machine_ID','Maintenance_Count','Machine_Age','Maintenance_Frequency']].drop_duplicates('Machine_ID').head()

## 4.7 Machine-level aggregates (excluded from model inputs)
Full-history machine aggregates can leak future information into predictions. We intentionally do not merge whole-period sensor averages or failure rates into the modeling dataset. If needed for descriptive reporting, calculate them only on a clearly defined historical window.


In [ ]:
# Do not create full-history machine aggregates here. In particular, never use
# Failure_Rate as an input feature because it is calculated from the target label.
# The model uses current sensor values, usage features and categorical bands.
df[['Machine_ID','Timestamp','Temperature','Vibration','Failure']].head()


## 4.8 Encode categorical bands as numeric (for modeling)

In [ ]:
df_model = pd.get_dummies(df, columns=['OpHours_Band','Age_Group'], drop_first=True)
df_model.shape

## 4.9 Save the feature-engineered dataset

In [ ]:
df_model.to_csv('../data/features_final.csv', index=False)
print('Saved features_final.csv, shape:', df_model.shape)
print('Total columns:', len(df_model.columns))